# RL Hybrid Control Environment

This notebook builds a measurement-aware `HybridCQEDEnv` directly inside the notebook and uses it as a template for simulator-trained bosonic-ancilla control studies in `cqed_sim`.

The walkthrough follows the intended digital-twin path:

- configure a reduced dispersive hybrid model and domain randomizer
- inspect the benchmark ladder and choose a task
- reset the environment with a deterministic seed
- run scripted and random rollouts
- evaluate the same controller under held-out randomization
- plot simulator-side diagnostics that remain hidden from the policy

## Scope

This notebook uses a coherent-state preparation task because it ships with a simple scripted baseline, but the same helper can be switched to other public benchmarks such as:

- `fock_state_preparation_task()`
- `storage_superposition_task()`
- `odd_cat_preparation_task()`
- `ancilla_storage_bell_task()`
- `conditional_phase_gate_task()`

The observation path below uses measurement classifier logits with history stacking, and the reward uses `build_reward_model("measurement_proxy")` so the example stays measurement-aware without requiring a full stochastic-trajectory workflow.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "cqed_sim").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current notebook working directory.")


REPO_ROOT = find_repo_root(Path.cwd())

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


from cqed_sim import (
    DomainRandomizer,
    HybridCQEDEnv,
    HybridEnvConfig,
    HybridSystemConfig,
    NormalPrior,
    PrimitiveActionSpace,
    QubitMeasurementSpec,
    ReducedDispersiveModelConfig,
    UniformPrior,
    benchmark_task_suite,
    build_observation_model,
    build_reward_model,
    coherent_state_preparation_task,
    odd_cat_preparation_task,
    storage_superposition_task,
    fock_state_preparation_task,
    conditional_phase_gate_task,
    ancilla_storage_bell_task,
 )


def build_environment(task=None) -> HybridCQEDEnv:
    nominal_chi = 2.0 * np.pi * (-2.25e6)
    system = HybridSystemConfig(
        regime="reduced_dispersive",
        reduced_model=ReducedDispersiveModelConfig(
            omega_c=2.0 * np.pi * 5.0e9,
            omega_q=2.0 * np.pi * 6.1e9,
            alpha=2.0 * np.pi * (-220.0e6),
            chi=nominal_chi,
            kerr=2.0 * np.pi * (-6.0e3),
            n_cav=10,
            n_tr=3,
        ),
        dt=4.0e-9,
        max_step=4.0e-9,
    )
    randomizer = DomainRandomizer(
        model_priors_train={
            "chi": NormalPrior(nominal_chi, 2.0 * np.pi * 0.08e6),
            "kerr": NormalPrior(2.0 * np.pi * (-6.0e3), 2.0 * np.pi * 1.5e3),
        },
        measurement_priors_train={
            "iq_sigma": UniformPrior(0.03, 0.07),
        },
        drift_priors_train={
            "storage_amplitude_scale": NormalPrior(1.0, 0.03, low=0.9, high=1.1),
        },
        model_priors_eval={
            "chi": UniformPrior(nominal_chi - 2.0 * np.pi * 0.12e6, nominal_chi + 2.0 * np.pi * 0.12e6),
            "kerr": UniformPrior(2.0 * np.pi * (-9.0e3), 2.0 * np.pi * (-3.0e3)),
        },
        measurement_priors_eval={
            "iq_sigma": UniformPrior(0.05, 0.09),
        },
        drift_priors_eval={
            "storage_amplitude_scale": UniformPrior(0.88, 1.12),
        },
    )
    measurement = QubitMeasurementSpec(
        shots=256,
        iq_sigma=0.05,
        confusion_matrix=np.asarray([[0.97, 0.04], [0.03, 0.96]], dtype=float),
    )
    action_space = PrimitiveActionSpace(primitives=("cavity_displacement", "wait", "measure"))
    selected_task = coherent_state_preparation_task(alpha=0.55 + 0.15j, duration=100.0e-9) if task is None else task
    return HybridCQEDEnv(
        HybridEnvConfig(
            system=system,
            task=selected_task,
            action_space=action_space,
            observation_model=build_observation_model(
                "measurement_classifier_logits",
                action_dim=action_space.shape[0],
                history_length=2,
            ),
            reward_model=build_reward_model("measurement_proxy"),
            randomizer=randomizer,
            randomization_mode="train",
            measurement_spec=measurement,
            auto_measurement=True,
            episode_horizon=2,
            seed=17,
            diagnostics_wigner_points=31,
        )
    )

In [ ]:
available_benchmarks = sorted(benchmark_task_suite().keys())
selected_task = coherent_state_preparation_task(alpha=0.55 + 0.15j, duration=100.0e-9)

env = build_environment(selected_task)

{
    "available_benchmarks": available_benchmarks,
    "selected_task": selected_task.name,
    "selected_task_description": selected_task.description,
    "suggested_alternatives": [
        odd_cat_preparation_task(alpha=1.0 + 0.0j).name,
        fock_state_preparation_task(1).name,
        storage_superposition_task().name,
        ancilla_storage_bell_task().name,
        conditional_phase_gate_task().name,
    ],
}

In [ ]:
initial_observation, initial_info = env.reset(seed=11)

{
    "observation_shape": tuple(int(value) for value in initial_observation.shape),
    "task_name": initial_info["task_name"],
    "task_kind": initial_info["task_kind"],
    "system_regime": initial_info["system_regime"],
    "initial_randomization": initial_info["randomization"],
    "initial_metrics": initial_info["metrics"],
}

In [ ]:
baseline = env.run_baseline(seed=11)

rng = np.random.default_rng(23)
random_actions = [env.config.action_space.sample(rng) for _ in range(2)]
random_rollout = env.rollout(random_actions, seed=23)

{
    "baseline_reward": baseline["total_reward"],
    "baseline_metrics": baseline["final_metrics"],
    "random_rollout_reward": random_rollout["total_reward"],
    "random_rollout_metrics": random_rollout["final_metrics"],
    "sampled_random_actions": [action.tolist() for action in random_actions],
}

In [ ]:
baseline = env.run_baseline(seed=11)
diagnostics = env.render_diagnostics()
evaluation = env.estimate_metrics(
    env.task.baseline_actions,
    seeds=(101, 102, 103, 104),
    randomization_mode="eval",
)

print(json.dumps(evaluation["summary"], indent=2))

photon_distribution = np.asarray(diagnostics.get("photon_number_distribution", []), dtype=float)
channel_payload = diagnostics.get("channels", {})
first_channel = next(iter(channel_payload.values()), None)
tlist = np.asarray(diagnostics.get("compiled_tlist", []), dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.0))
axes[0].bar(np.arange(photon_distribution.size), photon_distribution, color="#2a6f97")
axes[0].set_xlabel("storage Fock level")
axes[0].set_ylabel("population")
axes[0].set_title("Final photon-number distribution")

if first_channel is not None and tlist.size > 0:
    distorted = np.asarray(first_channel["distorted"], dtype=np.complex128)
    axes[1].plot(1.0e9 * tlist, distorted.real, label="I", linewidth=1.4)
    axes[1].plot(1.0e9 * tlist, distorted.imag, label="Q", linewidth=1.4)
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, "No compiled channel payload available.", ha="center", va="center")
axes[1].set_xlabel("time [ns]")
axes[1].set_ylabel("drive amplitude [rad/s]")
axes[1].set_title("Compiled distorted pulse")

fig.tight_layout()
plt.show()

{
    "segment_metadata": diagnostics["segment_metadata"],
    "pulse_summary": diagnostics["pulse_summary"],
    "evaluation_reward_summary": evaluation["summary"]["reward"],
}

## Notes

This notebook now exercises the environment directly. It is intended as a reusable template for future sim-to-real studies, not just as a report viewer.

Natural next variants to try inside `build_environment(...)`:

- swap in `odd_cat_preparation_task()` or `storage_superposition_task()` to study nonclassical cavity targets
- switch to `ParametricPulseActionSpace(family="hybrid_block")` for a continuous low-dimensional control interface
- replace the reduced model with a `FullPulseModelConfig` when you want a slower but more leakage-aware pulse-level study
- change the observation builder to `measurement_counts` or `measurement_outcome` if you want simpler partially observed policies